In [ ]:
import pandas as pd

# 1. Muat data
data_with_gemma = pd.read_csv('../short_gemma_bertscore.csv')
test_df = pd.read_csv('../test_df.csv')
train_df = pd.read_csv('../train_df.csv')

# 2. Ambil kolom kunci dan kolom target (description)
mapping_df = data_with_gemma[['graph', 'Essay', 'description', 'bertscore_precision', 'bertscore_recall', 'bertscore_f1']]

# 3. Petakan ke test_df
test_df_mapped = pd.merge(test_df, mapping_df, on=['graph', 'Essay'], how='left')

# 4. Petakan ke train_df
train_df_mapped = pd.merge(train_df, mapping_df, on=['graph', 'Essay'], how='left')

# 5. Simpan hasil ke file baru
test_df_mapped.to_csv('test_df_v2.csv', index=False)
train_df_mapped.to_csv('train_df_v2.csv', index=False)

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import cohen_kappa_score
from tqdm import tqdm
from bert_score import score
import gc
import random
import itertools
import warnings
import os

warnings.filterwarnings("ignore")

# ==========================================
# 0. KONFIGURASI AWAL & SEED
# ==========================================
def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        if hasattr(torch, 'xpu') and torch.xpu.is_available():
            torch.xpu.manual_seed(seed)
            torch.xpu.manual_seed_all(seed) # Berjaga-jaga jika ada multi-device
    except ImportError:
        print("⚠️ Warning: intel_extension_for_pytorch (ipex) tidak ditemukan.")
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(42)

def get_device():
    if torch.xpu.is_available(): return torch.device("xpu")
    if torch.cuda.is_available(): return torch.device("cuda")
    return torch.device("cpu")

DEVICE = get_device()
print(f"🔥 Using Device: {DEVICE}")

# KONFIGURASI EKSPERIMEN
TARGET_TRAIT = "persuasive"     # Bisa diubah sesuai kebutuhan ("clarity" atau "persuasive")
GROUP_COL = "graph"          # Kolom acuan untuk GroupKFold
MODEL_NAME = "roberta-base"
MAX_LEN = 512
BATCH_SIZE = 2
ACCUMULATION_STEPS = 4       # Effective batch size = 8
UNFREEZE_LAYERS = 6          # Dinaikkan agar model lebih pintar menangani Unseen Prompts
DATA_FILE = "short_gemma_bertscore.csv"
BEST_MODEL_SAVE_PATH = f"best_roberta_{TARGET_TRAIT}_final.pth"

# Parameter untuk eksperimen
K_FOLDS = 3                  

print(f"🎯 Fokus Pelatihan: Trait {TARGET_TRAIT.upper()}")

# ==========================================
# 1. PRAPEMROSESAN & BERTSCORE
# ==========================================
def prepare_data(csv_path):
    df = pd.read_csv(csv_path).dropna(subset=['description']).reset_index(drop=True)
    
    if 'bertscore_f1' not in df.columns:
        print("\n[*] Menghitung BERTScore antara Esai dan Description...")
        cands = df['Essay'].fillna("").tolist()
        refs = df['description'].fillna("").tolist()
        
        P, R, F1 = score(cands, refs, lang="en", verbose=True)
        df['bertscore_precision'] = P.numpy()
        df['bertscore_recall'] = R.numpy()
        df['bertscore_f1'] = F1.numpy()
        
        df.to_csv(csv_path, index=False)
        print(f"[*] Selesai! Data disimpan ke {csv_path}\n")
    return df

# ==========================================
# 2. DATASET CLASS
# ==========================================
import torch
from torch.utils.data import Dataset

class SingleTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, trait):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        self.questions = self.df['Question'].fillna("").values
        self.deplots = self.df['description'].fillna("").values
        self.essays = self.df['Essay'].fillna("").values
        self.bertscores = self.df['bertscore_f1'].fillna(0.0).values 
        
        if trait == "clarity":
            self.targets = self.df['argument_clarity(ground_truth)'].values
        elif trait == "persuasive":
            self.targets = self.df['justifying_persuasiveness(ground_truth)'].values
        else:
            raise ValueError("TARGET_TRAIT harus 'clarity' atau 'persuasive'")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        suffix_text = f"\n\nGraph Data: {self.deplots[index]} \n\nQuestion: {self.questions[index]}"
        suffix_tokens = self.tokenizer(suffix_text, add_special_tokens=False)['input_ids']
        
        prefix_text = f"Essay: {self.essays[index]}"
        prefix_tokens = self.tokenizer(prefix_text, add_special_tokens=False)['input_ids']
        
        num_special_tokens = 2
        
        # Pastikan suffix (Graph+Question) tidak melebihi max limit
        max_suffix_len = self.max_len - num_special_tokens
        if len(suffix_tokens) > max_suffix_len:
            suffix_tokens = suffix_tokens[:max_suffix_len]
            
        # Potong essay sesuai sisa token yang ada
        budget = self.max_len - len(suffix_tokens) - num_special_tokens
        prefix_tokens = prefix_tokens[:budget] if budget > 0 else []
            
        combined_tokens = prefix_tokens + suffix_tokens
        input_ids = [self.tokenizer.cls_token_id] + combined_tokens + [self.tokenizer.sep_token_id]
        
        attention_mask = [1] * len(input_ids)
        padding_length = self.max_len - len(input_ids)
        
        # Tambahkan padding jika total token masih di bawah max_len
        if padding_length > 0:
            input_ids += [self.tokenizer.pad_token_id] * padding_length
            attention_mask += [0] * padding_length
            
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'bertscore': torch.tensor(self.bertscores[index], dtype=torch.float),
            'target': torch.tensor(self.targets[index], dtype=torch.float)
        }

# ==========================================
# 3. MODEL ARCHITECTURE
# ==========================================
class RobertaAES(nn.Module):
    def __init__(self, model_name, dropout_rate=0.3):
        super(RobertaAES, self).__init__()
        
        self.transformer = AutoModel.from_pretrained(model_name)
        self.drop = nn.Dropout(p=dropout_rate) 
        
        for param in self.transformer.parameters():
            param.requires_grad = False
            
        for layer in self.transformer.encoder.layer[-UNFREEZE_LAYERS:]:
            for param in layer.parameters(): 
                param.requires_grad = True

        hidden_size = self.transformer.config.hidden_size
        self.score_head = nn.Linear(hidden_size + 1, 1)

    def forward(self, input_ids, attention_mask, bertscore):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        dropped_output = self.drop(pooled_output)
        
        combined_features = torch.cat((dropped_output, bertscore), dim=1)
        score = self.score_head(combined_features)
        return score

# Fungsi Evaluasi QWK
def evaluate_qwk(model, dataloader):
    model.eval()
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for b in dataloader:
            input_ids = b['input_ids'].to(DEVICE)
            attention_mask = b['attention_mask'].to(DEVICE)
            bertscore = b['bertscore'].to(DEVICE).unsqueeze(1)
            target = b['target'].to(DEVICE).unsqueeze(1)
            
            if DEVICE.type in ["cuda", "xpu"]:
                with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16 if DEVICE.type == "xpu" else torch.float16):
                    pred = model(input_ids, attention_mask, bertscore)
            else:
                pred = model(input_ids, attention_mask, bertscore)
                
            all_preds.extend(pred.float().cpu().numpy().flatten())
            all_targets.extend(target.float().cpu().numpy().flatten())
            
    pred_classes = np.round(np.array(all_preds) * 2).astype(int)
    true_classes = np.round(np.array(all_targets) * 2).astype(int)
    return cohen_kappa_score(true_classes, pred_classes, weights='quadratic')

# ==========================================
# 4. MAIN EXECUTION PIPELINE
# ==========================================
if __name__ == "__main__":
    # --- STAGE 1: PEMBAGIAN DATA (HOLD-OUT TEST SET) ---
    df = prepare_data(DATA_FILE)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    # CEK TRAIN/TEST SPLIT EKSISTING (Agar Test Set konsisten dengan percobaan lain)
    if os.path.exists('train_df_v2.csv') and os.path.exists('test_df_v2.csv'):
        print("🔄 Menemukan train_df_v2.csv dan test_df_v2.csv! Memuat data eksisting agar Test Set murni tetap sama...")
        df_train = pd.read_csv('train_df_v2.csv')
        df_test = pd.read_csv('test_df_v2.csv')
    else:
        print("⚠️ File split belum ada. Melakukan GroupShuffleSplit baru dan menyimpannya...")
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, test_idx = next(gss.split(df, groups=df[GROUP_COL]))
        
        df_train = df.iloc[train_idx].reset_index(drop=True)
        df_test = df.iloc[test_idx].reset_index(drop=True)
        
        df_train.to_csv('train_df.csv', index=False)
        df_test.to_csv('test_df.csv', index=False)
    
    test_dataset = SingleTaskDataset(df_test, tokenizer, MAX_LEN, TARGET_TRAIT)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    print(f"\n✅ Data Total: {len(df)} | Train Murni: {len(df_train)} | Test Murni: {len(df_test)}")
    
    # --- STAGE 2: GRID SEARCH + GROUP K-FOLD DENGAN AUTO-RESUME ---
    print("\n🚀 STAGE 2: MENCARI PARAMETER TERBAIK (CROSS-VALIDATION)")
    
    param_grid = {
        'learning_rate': [1e-5, 2e-5, 3e-5], 
        'dropout_rate': [0.1, 0.2],    
        'epochs': [10, 15]            
    }
    
    keys = param_grid.keys()
    combinations = list(itertools.product(*param_grid.values()))
    
    best_cv_score = -1.0
    best_params = None
    semua_hasil = []
    kombo_selesai = []
    file_hasil = f'grid_search_{TARGET_TRAIT}_results.csv'
    
    gkf = GroupKFold(n_splits=K_FOLDS)
    scaler = torch.amp.GradScaler(DEVICE.type) if DEVICE.type in ["cuda", "xpu"] else None

    # FITUR AUTO-RESUME
    if os.path.exists(file_hasil):
        try:
            df_lama = pd.read_csv(file_hasil)
            if not df_lama.empty:
                kombo_selesai = df_lama['combo_index'].tolist()
                semua_hasil = df_lama.to_dict('records') 
                
                # Cari best score dari riwayat sebelumnya
                best_idx = df_lama['Mean_QWK'].idxmax()
                best_cv_score = df_lama.loc[best_idx, 'Mean_QWK']
                best_params = {
                    'learning_rate': df_lama.loc[best_idx, 'learning_rate'],
                    'dropout_rate': df_lama.loc[best_idx, 'dropout_rate'],
                    'epochs': int(df_lama.loc[best_idx, 'epochs'])
                }
                print(f"🔄 Menemukan riwayat! Melewati {len(kombo_selesai)} kombinasi yang sudah beres...")
                print(f"⭐ Best skor sementara dari riwayat: {best_cv_score:.4f}")
        except Exception as e:
            print(f"⚠️ Gagal membaca {file_hasil}, memulai dari awal. Error: {e}")

    for i, combo in enumerate(combinations):
        # SKIP KOMBINASI JIKA SUDAH SELESAI
        if i in kombo_selesai:
            print(f"⏩ Skip Kombinasi [{i+1}/{len(combinations)}], sudah selesai sebelumnya.")
            continue
            
        params = dict(zip(keys, combo))
        print(f"\n⚙️ [{i+1}/{len(combinations)}] Uji: LR={params['learning_rate']} | Drop={params['dropout_rate']} | Epochs={params['epochs']}")
        
        fold_qwks = []
        for fold, (f_train_idx, f_val_idx) in enumerate(gkf.split(df_train, groups=df_train[GROUP_COL])):
            current_seed = 42 + (i * 10) + fold
            set_seed(current_seed)
            
            g = torch.Generator()
            g.manual_seed(current_seed)

            if 'model' in locals(): 
                model.to('cpu') 
                del model
            if 'optimizer' in locals(): 
                del optimizer
                
            gc.collect()
            if DEVICE.type == "xpu": torch.xpu.empty_cache()
            elif DEVICE.type == "cuda": torch.cuda.empty_cache()
            
            f_train_df = df_train.iloc[f_train_idx]
            f_val_df = df_train.iloc[f_val_idx]
            
            f_train_loader = DataLoader(SingleTaskDataset(f_train_df, tokenizer, MAX_LEN, TARGET_TRAIT), batch_size=BATCH_SIZE, shuffle=True, generator=g)
            f_val_loader = DataLoader(SingleTaskDataset(f_val_df, tokenizer, MAX_LEN, TARGET_TRAIT), batch_size=BATCH_SIZE, shuffle=False)
            
            model = RobertaAES(MODEL_NAME, dropout_rate=params['dropout_rate']).to(DEVICE)
            optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=params['learning_rate'])
            criterion = nn.MSELoss()
            
            for epoch in range(params['epochs']):
                model.train()
                for step, batch in enumerate(f_train_loader):
                    ids = batch['input_ids'].to(DEVICE)
                    mask = batch['attention_mask'].to(DEVICE)
                    b_score = batch['bertscore'].to(DEVICE).unsqueeze(1)
                    tgt = batch['target'].to(DEVICE).unsqueeze(1)
                    
                    if scaler:
                        with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16 if DEVICE.type == "xpu" else torch.float16):
                            pred = model(ids, mask, b_score)
                            loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
                        scaler.scale(loss).backward()
                        
                        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(f_train_loader):
                            scaler.step(optimizer)
                            scaler.update()
                            optimizer.zero_grad()
                    else:
                        pred = model(ids, mask, b_score)
                        loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
                        loss.backward()
                        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(f_train_loader):
                            optimizer.step()
                            optimizer.zero_grad()

                    del ids, mask, b_score, tgt, pred, loss
                if DEVICE.type == "xpu": torch.xpu.empty_cache()
                            
            fold_qwk = evaluate_qwk(model, f_val_loader)
            fold_qwks.append(fold_qwk)
            print(f"   Fold {fold+1} QWK: {fold_qwk:.4f}")

            del model, optimizer
            gc.collect()
            
        mean_cv_qwk = np.mean(fold_qwks)
        print(f"📊 Rata-rata QWK: {mean_cv_qwk:.4f}")
        
        semua_hasil.append({
            'combo_index': i,
            'learning_rate': params['learning_rate'], 
            'dropout_rate': params['dropout_rate'], 
            'epochs': params['epochs'], 
            'Mean_QWK': mean_cv_qwk
        })
        
        if mean_cv_qwk > best_cv_score:
            print("🌟 Kombinasi terbaik sejauh ini!")
            best_cv_score = mean_cv_qwk
            best_params = params

        # SIMPAN HASIL KE CSV SETELAH 1 KOMBINASI SELESAI
        pd.DataFrame(semua_hasil).to_csv(file_hasil, index=False)
        print(f"💾 Progress aman! Tersimpan ke {file_hasil}\n")

    print("\n" + "="*50)
    print("🏆 HASIL GRID SEARCH SELESAI!")
    print(pd.DataFrame(semua_hasil).to_markdown(index=False))
    print(f"🌟 Parameter Terbaik: {best_params}")
    print("="*50)

    # --- STAGE 3: FINAL TRAINING ---
    print("\n🔥 STAGE 3: MELATIH FINAL MODEL (MENGGUNAKAN 100% DATA TRAIN)...")
    gc.collect()
    if DEVICE.type == "xpu": torch.xpu.empty_cache()
    elif DEVICE.type == "cuda": torch.cuda.empty_cache()

    set_seed(42)
    g_final = torch.Generator()
    g_final.manual_seed(42)
    
    full_train_loader = DataLoader(SingleTaskDataset(df_train, tokenizer, MAX_LEN, TARGET_TRAIT), batch_size=BATCH_SIZE, shuffle=True, generator=g_final)
    
    final_model = RobertaAES(MODEL_NAME, dropout_rate=best_params['dropout_rate']).to(DEVICE)
    final_optimizer = AdamW(filter(lambda p: p.requires_grad, final_model.parameters()), lr=best_params['learning_rate'])
    criterion = nn.MSELoss()
    
    # Menggunakan epochs terbaik yang ditemukan pada Grid Search
    final_epochs = best_params['epochs']
    
    for epoch in range(final_epochs):
        final_model.train()
        loop = tqdm(full_train_loader, desc=f"Final Epoch {epoch+1}/{final_epochs}")
        for step, batch in enumerate(loop):
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            b_score = batch['bertscore'].to(DEVICE).unsqueeze(1)
            tgt = batch['target'].to(DEVICE).unsqueeze(1)
            
            if scaler:
                with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16 if DEVICE.type == "xpu" else torch.float16):
                    pred = final_model(ids, mask, b_score)
                    loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
                scaler.scale(loss).backward()
                
                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(full_train_loader):
                    scaler.step(final_optimizer)
                    scaler.update()
                    final_optimizer.zero_grad()
            else:
                pred = final_model(ids, mask, b_score)
                loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
                loss.backward()
                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(full_train_loader):
                    final_optimizer.step()
                    final_optimizer.zero_grad()
                    
            loop.set_postfix({'Loss': f"{(loss.item() * ACCUMULATION_STEPS):.4f}"})

    torch.save(final_model.state_dict(), BEST_MODEL_SAVE_PATH)
    print(f"✅ Final Model Tersimpan: {BEST_MODEL_SAVE_PATH}")

    # ==========================================
    # --- STAGE 4: EXTRACTING PREDICTIONS TO CSV ---
    # ==========================================
    print("\n⚙️ STAGE 4: EXTRACTING ROBERTA PREDICTIONS (MENYIMPAN KE CSV)...")

    final_model.eval()

    # Buat dataloader train khusus tanpa shuffle agar urutannya sesuai dengan df_train
    train_extract_loader = DataLoader(SingleTaskDataset(df_train, tokenizer, MAX_LEN, TARGET_TRAIT), batch_size=BATCH_SIZE, shuffle=False)

    def extract_predictions(loader, desc):
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for batch in tqdm(loader, desc=desc):
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                b_score = batch['bertscore'].to(DEVICE).unsqueeze(1)
                
                # Biarkan target di CPU untuk dimasukkan ke DataFrame
                tgt = batch['target'].numpy() 

                # Forward pass dengan autocast jika pakai GPU/XPU
                if DEVICE.type in ["cuda", "xpu"]:
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16 if DEVICE.type == "xpu" else torch.float16):
                        preds = final_model(ids, mask, b_score)
                else:
                    preds = final_model(ids, mask, b_score)

                all_preds.extend(preds.float().cpu().numpy().flatten())
                all_targets.extend(tgt.flatten())
                
        return all_preds, all_targets

    # 1. Ekstraksi Train Set
    train_preds, train_targets = extract_predictions(train_extract_loader, "Extracting Train Preds")
    df_train_out = pd.DataFrame({
        'ground_truth': train_targets, 
        'prediction': train_preds,
        'rounded_prediction': np.round(np.array(train_preds) * 2) / 2
    })
    train_csv_name = f'roberta_train_{TARGET_TRAIT}.csv'
    df_train_out.to_csv(train_csv_name, index=False)

    # 2. Ekstraksi Test Set
    test_preds, test_targets = extract_predictions(test_loader, "Extracting Test Preds")
    df_test_out = pd.DataFrame({
        'ground_truth': test_targets, 
        'prediction': test_preds,
        'rounded_prediction': np.round(np.array(test_preds) * 2) / 2
    })
    test_csv_name = f'roberta_test_{TARGET_TRAIT}.csv'
    df_test_out.to_csv(test_csv_name, index=False)

    print(f"\n✅ Prediksi Train Set disimpan ke: {train_csv_name}")
    print(f"✅ Prediksi Test Set disimpan ke : {test_csv_name}")


    # ==========================================
    # --- STAGE 5: FINAL EVALUATION (MOMEN KEBENARAN) ---
    # ==========================================
    print("\n🏆 STAGE 5: EVALUASI AKHIR")
    
    # Menghitung QWK langsung dari hasil ekstraksi, tidak perlu memanggil model lagi
    pred_classes = np.round(np.array(test_preds) * 2).astype(int)
    true_classes = np.round(np.array(test_targets) * 2).astype(int)
    
    final_qwk = cohen_kappa_score(true_classes, pred_classes, weights='quadratic')

    print("="*50)
    print(f"🌟 SKOR QWK FINAL (PADA TEST SET MURNI) UNTUK TRAIT '{TARGET_TRAIT.upper()}': {final_qwk:.4f} 🌟")
    print("="*50)

c:\Users\gabyg\miniconda3\envs\skripsi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔥 Using Device: xpu
🎯 Fokus Pelatihan: Trait PERSUASIVE
🔄 Menemukan train_df_v2.csv dan test_df_v2.csv! Memuat data eksisting agar Test Set murni tetap sama...

✅ Data Total: 1054 | Train Murni: 867 | Test Murni: 187

🚀 STAGE 2: MENCARI PARAMETER TERBAIK (CROSS-VALIDATION)

⚙️ [1/12] Uji: LR=1e-05 | Drop=0.1 | Epochs=10


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 301.69it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   Fold 1 QWK: 0.4000


Loading weights: 100%|██████████| 197/197 [00:01<00:00, 137.68it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

   Fold 3 QWK: 0.5305
📊 Rata-rata QWK: 0.4021
💾 Progress aman! Tersimpan ke grid_search_persuasive_results.csv


🏆 HASIL GRID SEARCH SELESAI!
|   combo_index |   learning_rate |   dropout_rate |   epochs |   Mean_QWK |
|--------------:|----------------:|---------------:|---------:|-----------:|
|             0 |           1e-05 |            0.1 |       10 |   0.381703 |
|             1 |           1e-05 |            0.1 |       15 |   0.430169 |
|             2 |           1e-05 |            0.2 |       10 |   0.391767 |
|             3 |           1e-05 |            0.2 |       15 |   0.421963 |
|             4 |           2e-05 |            0.1 |       10 |   0.343333 |
|             5 |           2e-05 |            0.1 |       15 |   0.347218 |
|             6 |           2e-05 |            0.2 |       10 |   0.439434 |
|             7 |           2e-05 |            0.2 |       15 |   0.398575 |
|             8 |           3e-05 |            0.1 |       10 |   0.344879 |
|          

Loading weights: 100%|██████████| 197/197 [00:01<00:00, 142.50it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Final Epoch 10/10: 100%|██████████| 434/434 [03:03<00:00,  2.36it/s, Loss=0.0198]


✅ Final Model Tersimpan: best_roberta_persuasive_final.pth

⚙️ STAGE 4: EXTRACTING ROBERTA PREDICTIONS (MENYIMPAN KE CSV)...


Extracting Test Preds: 100%|██████████| 94/94 [00:13<00:00,  6.92it/s]



✅ Prediksi Train Set disimpan ke: roberta_train_persuasive.csv
✅ Prediksi Test Set disimpan ke : roberta_test_persuasive.csv

🏆 STAGE 5: EVALUASI AKHIR
🌟 SKOR QWK FINAL (PADA TEST SET MURNI) UNTUK TRAIT 'PERSUASIVE': 0.4622 🌟


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import cohen_kappa_score
from tqdm import tqdm
from bert_score import score
import gc
import random
import itertools
import warnings
import os

warnings.filterwarnings("ignore")

# ==========================================
# 0. KONFIGURASI AWAL & SEED
# ==========================================
def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        if hasattr(torch, 'xpu') and torch.xpu.is_available():
            torch.xpu.manual_seed(seed)
            torch.xpu.manual_seed_all(seed) # Berjaga-jaga jika ada multi-device
    except ImportError:
        print("⚠️ Warning: intel_extension_for_pytorch (ipex) tidak ditemukan.")
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(42)

def get_device():
    if torch.xpu.is_available(): return torch.device("xpu")
    if torch.cuda.is_available(): return torch.device("cuda")
    return torch.device("cpu")

DEVICE = get_device()
print(f"🔥 Using Device: {DEVICE}")

# KONFIGURASI EKSPERIMEN
TARGET_TRAIT = "clarity"     # Bisa diubah sesuai kebutuhan ("clarity" atau "persuasive")
GROUP_COL = "graph"          # Kolom acuan untuk GroupKFold
MODEL_NAME = "roberta-base"
MAX_LEN = 512
BATCH_SIZE = 2
ACCUMULATION_STEPS = 4       # Effective batch size = 8
UNFREEZE_LAYERS = 6          # Dinaikkan agar model lebih pintar menangani Unseen Prompts
DATA_FILE = "short_gemma_bertscore.csv"
BEST_MODEL_SAVE_PATH = f"best_roberta_{TARGET_TRAIT}_final.pth"

# Parameter untuk eksperimen
K_FOLDS = 3                  

print(f"🎯 Fokus Pelatihan: Trait {TARGET_TRAIT.upper()}")

# ==========================================
# 1. PRAPEMROSESAN & BERTSCORE
# ==========================================
def prepare_data(csv_path):
    df = pd.read_csv(csv_path).dropna(subset=['description']).reset_index(drop=True)
    
    if 'bertscore_f1' not in df.columns:
        print("\n[*] Menghitung BERTScore antara Esai dan Description...")
        cands = df['Essay'].fillna("").tolist()
        refs = df['description'].fillna("").tolist()
        
        P, R, F1 = score(cands, refs, lang="en", verbose=True)
        df['bertscore_precision'] = P.numpy()
        df['bertscore_recall'] = R.numpy()
        df['bertscore_f1'] = F1.numpy()
        
        df.to_csv(csv_path, index=False)
        print(f"[*] Selesai! Data disimpan ke {csv_path}\n")
    return df

# ==========================================
# 2. DATASET CLASS
# ==========================================
import torch
from torch.utils.data import Dataset

class SingleTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, trait):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        self.questions = self.df['Question'].fillna("").values
        self.deplots = self.df['description'].fillna("").values
        self.essays = self.df['Essay'].fillna("").values
        self.bertscores = self.df['bertscore_f1'].fillna(0.0).values 
        
        if trait == "clarity":
            self.targets = self.df['argument_clarity(ground_truth)'].values
        elif trait == "persuasive":
            self.targets = self.df['justifying_persuasiveness(ground_truth)'].values
        else:
            raise ValueError("TARGET_TRAIT harus 'clarity' atau 'persuasive'")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        suffix_text = f"\n\nGraph Data: {self.deplots[index]} \n\nQuestion: {self.questions[index]}"
        suffix_tokens = self.tokenizer(suffix_text, add_special_tokens=False)['input_ids']
        
        prefix_text = f"Essay: {self.essays[index]}"
        prefix_tokens = self.tokenizer(prefix_text, add_special_tokens=False)['input_ids']
        
        num_special_tokens = 2
        
        # Pastikan suffix (Graph+Question) tidak melebihi max limit
        max_suffix_len = self.max_len - num_special_tokens
        if len(suffix_tokens) > max_suffix_len:
            suffix_tokens = suffix_tokens[:max_suffix_len]
            
        # Potong essay sesuai sisa token yang ada
        budget = self.max_len - len(suffix_tokens) - num_special_tokens
        prefix_tokens = prefix_tokens[:budget] if budget > 0 else []
            
        combined_tokens = prefix_tokens + suffix_tokens
        input_ids = [self.tokenizer.cls_token_id] + combined_tokens + [self.tokenizer.sep_token_id]
        
        attention_mask = [1] * len(input_ids)
        padding_length = self.max_len - len(input_ids)
        
        # Tambahkan padding jika total token masih di bawah max_len
        if padding_length > 0:
            input_ids += [self.tokenizer.pad_token_id] * padding_length
            attention_mask += [0] * padding_length
            
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'bertscore': torch.tensor(self.bertscores[index], dtype=torch.float),
            'target': torch.tensor(self.targets[index], dtype=torch.float)
        }

# ==========================================
# 3. MODEL ARCHITECTURE
# ==========================================
class RobertaAES(nn.Module):
    def __init__(self, model_name, dropout_rate=0.3):
        super(RobertaAES, self).__init__()
        
        self.transformer = AutoModel.from_pretrained(model_name)
        self.drop = nn.Dropout(p=dropout_rate) 
        
        for param in self.transformer.parameters():
            param.requires_grad = False
            
        for layer in self.transformer.encoder.layer[-UNFREEZE_LAYERS:]:
            for param in layer.parameters(): 
                param.requires_grad = True

        hidden_size = self.transformer.config.hidden_size
        self.score_head = nn.Linear(hidden_size + 1, 1)

    def forward(self, input_ids, attention_mask, bertscore):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        dropped_output = self.drop(pooled_output)
        
        combined_features = torch.cat((dropped_output, bertscore), dim=1)
        score = self.score_head(combined_features)
        return score

# Fungsi Evaluasi QWK
def evaluate_qwk(model, dataloader):
    model.eval()
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for b in dataloader:
            input_ids = b['input_ids'].to(DEVICE)
            attention_mask = b['attention_mask'].to(DEVICE)
            bertscore = b['bertscore'].to(DEVICE).unsqueeze(1)
            target = b['target'].to(DEVICE).unsqueeze(1)
            
            if DEVICE.type in ["cuda", "xpu"]:
                with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16 if DEVICE.type == "xpu" else torch.float16):
                    pred = model(input_ids, attention_mask, bertscore)
            else:
                pred = model(input_ids, attention_mask, bertscore)
                
            all_preds.extend(pred.float().cpu().numpy().flatten())
            all_targets.extend(target.float().cpu().numpy().flatten())
            
    pred_classes = np.round(np.array(all_preds) * 2).astype(int)
    true_classes = np.round(np.array(all_targets) * 2).astype(int)
    return cohen_kappa_score(true_classes, pred_classes, weights='quadratic')

# ==========================================
# 4. MAIN EXECUTION PIPELINE
# ==========================================
if __name__ == "__main__":
    # --- STAGE 1: PEMBAGIAN DATA (HOLD-OUT TEST SET) ---
    df = prepare_data(DATA_FILE)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    # CEK TRAIN/TEST SPLIT EKSISTING (Agar Test Set konsisten dengan percobaan lain)
    if os.path.exists('train_df_v2.csv') and os.path.exists('test_df_v2.csv'):
        print("🔄 Menemukan train_df_v2.csv dan test_df_v2.csv! Memuat data eksisting agar Test Set murni tetap sama...")
        df_train = pd.read_csv('train_df_v2.csv')
        df_test = pd.read_csv('test_df_v2.csv')
    else:
        print("⚠️ File split belum ada. Melakukan GroupShuffleSplit baru dan menyimpannya...")
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, test_idx = next(gss.split(df, groups=df[GROUP_COL]))
        
        df_train = df.iloc[train_idx].reset_index(drop=True)
        df_test = df.iloc[test_idx].reset_index(drop=True)
        
        df_train.to_csv('train_df.csv', index=False)
        df_test.to_csv('test_df.csv', index=False)
    
    test_dataset = SingleTaskDataset(df_test, tokenizer, MAX_LEN, TARGET_TRAIT)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    print(f"\n✅ Data Total: {len(df)} | Train Murni: {len(df_train)} | Test Murni: {len(df_test)}")
    
    # --- STAGE 2: GRID SEARCH + GROUP K-FOLD DENGAN AUTO-RESUME ---
    print("\n🚀 STAGE 2: MENCARI PARAMETER TERBAIK (CROSS-VALIDATION)")
    
    param_grid = {
        'learning_rate': [1e-5, 2e-5, 3e-5], 
        'dropout_rate': [0.1, 0.2],    
        'epochs': [10, 15]            
    }
    
    keys = param_grid.keys()
    combinations = list(itertools.product(*param_grid.values()))
    
    best_cv_score = -1.0
    best_params = None
    semua_hasil = []
    kombo_selesai = []
    file_hasil = f'grid_search_{TARGET_TRAIT}_results.csv'
    
    gkf = GroupKFold(n_splits=K_FOLDS)
    scaler = torch.amp.GradScaler(DEVICE.type) if DEVICE.type in ["cuda", "xpu"] else None

    # FITUR AUTO-RESUME
    if os.path.exists(file_hasil):
        try:
            df_lama = pd.read_csv(file_hasil)
            if not df_lama.empty:
                kombo_selesai = df_lama['combo_index'].tolist()
                semua_hasil = df_lama.to_dict('records') 
                
                # Cari best score dari riwayat sebelumnya
                best_idx = df_lama['Mean_QWK'].idxmax()
                best_cv_score = df_lama.loc[best_idx, 'Mean_QWK']
                best_params = {
                    'learning_rate': df_lama.loc[best_idx, 'learning_rate'],
                    'dropout_rate': df_lama.loc[best_idx, 'dropout_rate'],
                    'epochs': int(df_lama.loc[best_idx, 'epochs'])
                }
                print(f"🔄 Menemukan riwayat! Melewati {len(kombo_selesai)} kombinasi yang sudah beres...")
                print(f"⭐ Best skor sementara dari riwayat: {best_cv_score:.4f}")
        except Exception as e:
            print(f"⚠️ Gagal membaca {file_hasil}, memulai dari awal. Error: {e}")

    for i, combo in enumerate(combinations):
        # SKIP KOMBINASI JIKA SUDAH SELESAI
        if i in kombo_selesai:
            print(f"⏩ Skip Kombinasi [{i+1}/{len(combinations)}], sudah selesai sebelumnya.")
            continue
            
        params = dict(zip(keys, combo))
        print(f"\n⚙️ [{i+1}/{len(combinations)}] Uji: LR={params['learning_rate']} | Drop={params['dropout_rate']} | Epochs={params['epochs']}")
        
        fold_qwks = []
        for fold, (f_train_idx, f_val_idx) in enumerate(gkf.split(df_train, groups=df_train[GROUP_COL])):
            current_seed = 42 + (i * 10) + fold
            set_seed(current_seed)
            
            g = torch.Generator()
            g.manual_seed(current_seed)

            if 'model' in locals(): 
                model.to('cpu') 
                del model
            if 'optimizer' in locals(): 
                del optimizer
                
            gc.collect()
            if DEVICE.type == "xpu": torch.xpu.empty_cache()
            elif DEVICE.type == "cuda": torch.cuda.empty_cache()
            
            f_train_df = df_train.iloc[f_train_idx]
            f_val_df = df_train.iloc[f_val_idx]
            
            f_train_loader = DataLoader(SingleTaskDataset(f_train_df, tokenizer, MAX_LEN, TARGET_TRAIT), batch_size=BATCH_SIZE, shuffle=True, generator=g)
            f_val_loader = DataLoader(SingleTaskDataset(f_val_df, tokenizer, MAX_LEN, TARGET_TRAIT), batch_size=BATCH_SIZE, shuffle=False)
            
            model = RobertaAES(MODEL_NAME, dropout_rate=params['dropout_rate']).to(DEVICE)
            optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=params['learning_rate'])
            criterion = nn.MSELoss()
            
            for epoch in range(params['epochs']):
                model.train()
                for step, batch in enumerate(f_train_loader):
                    ids = batch['input_ids'].to(DEVICE)
                    mask = batch['attention_mask'].to(DEVICE)
                    b_score = batch['bertscore'].to(DEVICE).unsqueeze(1)
                    tgt = batch['target'].to(DEVICE).unsqueeze(1)
                    
                    if scaler:
                        with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16 if DEVICE.type == "xpu" else torch.float16):
                            pred = model(ids, mask, b_score)
                            loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
                        scaler.scale(loss).backward()
                        
                        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(f_train_loader):
                            scaler.step(optimizer)
                            scaler.update()
                            optimizer.zero_grad()
                    else:
                        pred = model(ids, mask, b_score)
                        loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
                        loss.backward()
                        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(f_train_loader):
                            optimizer.step()
                            optimizer.zero_grad()

                    del ids, mask, b_score, tgt, pred, loss
                if DEVICE.type == "xpu": torch.xpu.empty_cache()
                            
            fold_qwk = evaluate_qwk(model, f_val_loader)
            fold_qwks.append(fold_qwk)
            print(f"   Fold {fold+1} QWK: {fold_qwk:.4f}")

            del model, optimizer
            gc.collect()
            
        mean_cv_qwk = np.mean(fold_qwks)
        print(f"📊 Rata-rata QWK: {mean_cv_qwk:.4f}")
        
        semua_hasil.append({
            'combo_index': i,
            'learning_rate': params['learning_rate'], 
            'dropout_rate': params['dropout_rate'], 
            'epochs': params['epochs'], 
            'Mean_QWK': mean_cv_qwk
        })
        
        if mean_cv_qwk > best_cv_score:
            print("🌟 Kombinasi terbaik sejauh ini!")
            best_cv_score = mean_cv_qwk
            best_params = params

        # SIMPAN HASIL KE CSV SETELAH 1 KOMBINASI SELESAI
        pd.DataFrame(semua_hasil).to_csv(file_hasil, index=False)
        print(f"💾 Progress aman! Tersimpan ke {file_hasil}\n")

    print("\n" + "="*50)
    print("🏆 HASIL GRID SEARCH SELESAI!")
    print(pd.DataFrame(semua_hasil).to_markdown(index=False))
    print(f"🌟 Parameter Terbaik: {best_params}")
    print("="*50)

    # --- STAGE 3: FINAL TRAINING ---
    print("\n🔥 STAGE 3: MELATIH FINAL MODEL (MENGGUNAKAN 100% DATA TRAIN)...")
    gc.collect()
    if DEVICE.type == "xpu": torch.xpu.empty_cache()
    elif DEVICE.type == "cuda": torch.cuda.empty_cache()

    set_seed(42)
    g_final = torch.Generator()
    g_final.manual_seed(42)
    
    full_train_loader = DataLoader(SingleTaskDataset(df_train, tokenizer, MAX_LEN, TARGET_TRAIT), batch_size=BATCH_SIZE, shuffle=True, generator=g_final)
    
    final_model = RobertaAES(MODEL_NAME, dropout_rate=best_params['dropout_rate']).to(DEVICE)
    final_optimizer = AdamW(filter(lambda p: p.requires_grad, final_model.parameters()), lr=best_params['learning_rate'])
    criterion = nn.MSELoss()
    
    # Menggunakan epochs terbaik yang ditemukan pada Grid Search
    final_epochs = best_params['epochs']
    
    for epoch in range(final_epochs):
        final_model.train()
        loop = tqdm(full_train_loader, desc=f"Final Epoch {epoch+1}/{final_epochs}")
        for step, batch in enumerate(loop):
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            b_score = batch['bertscore'].to(DEVICE).unsqueeze(1)
            tgt = batch['target'].to(DEVICE).unsqueeze(1)
            
            if scaler:
                with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16 if DEVICE.type == "xpu" else torch.float16):
                    pred = final_model(ids, mask, b_score)
                    loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
                scaler.scale(loss).backward()
                
                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(full_train_loader):
                    scaler.step(final_optimizer)
                    scaler.update()
                    final_optimizer.zero_grad()
            else:
                pred = final_model(ids, mask, b_score)
                loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
                loss.backward()
                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(full_train_loader):
                    final_optimizer.step()
                    final_optimizer.zero_grad()
                    
            loop.set_postfix({'Loss': f"{(loss.item() * ACCUMULATION_STEPS):.4f}"})

    torch.save(final_model.state_dict(), BEST_MODEL_SAVE_PATH)
    print(f"✅ Final Model Tersimpan: {BEST_MODEL_SAVE_PATH}")

    # ==========================================
    # --- STAGE 4: EXTRACTING PREDICTIONS TO CSV ---
    # ==========================================
    print("\n⚙️ STAGE 4: EXTRACTING ROBERTA PREDICTIONS (MENYIMPAN KE CSV)...")

    final_model.eval()

    # Buat dataloader train khusus tanpa shuffle agar urutannya sesuai dengan df_train
    train_extract_loader = DataLoader(SingleTaskDataset(df_train, tokenizer, MAX_LEN, TARGET_TRAIT), batch_size=BATCH_SIZE, shuffle=False)

    def extract_predictions(loader, desc):
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for batch in tqdm(loader, desc=desc):
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                b_score = batch['bertscore'].to(DEVICE).unsqueeze(1)
                
                # Biarkan target di CPU untuk dimasukkan ke DataFrame
                tgt = batch['target'].numpy() 

                # Forward pass dengan autocast jika pakai GPU/XPU
                if DEVICE.type in ["cuda", "xpu"]:
                    with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16 if DEVICE.type == "xpu" else torch.float16):
                        preds = final_model(ids, mask, b_score)
                else:
                    preds = final_model(ids, mask, b_score)

                all_preds.extend(preds.float().cpu().numpy().flatten())
                all_targets.extend(tgt.flatten())
                
        return all_preds, all_targets

    # 1. Ekstraksi Train Set
    train_preds, train_targets = extract_predictions(train_extract_loader, "Extracting Train Preds")
    df_train_out = pd.DataFrame({
        'ground_truth': train_targets, 
        'prediction': train_preds,
        'rounded_prediction': np.round(np.array(train_preds) * 2) / 2
    })
    train_csv_name = f'roberta_train_{TARGET_TRAIT}.csv'
    df_train_out.to_csv(train_csv_name, index=False)

    # 2. Ekstraksi Test Set
    test_preds, test_targets = extract_predictions(test_loader, "Extracting Test Preds")
    df_test_out = pd.DataFrame({
        'ground_truth': test_targets, 
        'prediction': test_preds,
        'rounded_prediction': np.round(np.array(test_preds) * 2) / 2
    })
    test_csv_name = f'roberta_test_{TARGET_TRAIT}.csv'
    df_test_out.to_csv(test_csv_name, index=False)

    print(f"\n✅ Prediksi Train Set disimpan ke: {train_csv_name}")
    print(f"✅ Prediksi Test Set disimpan ke : {test_csv_name}")


    # ==========================================
    # --- STAGE 5: FINAL EVALUATION (MOMEN KEBENARAN) ---
    # ==========================================
    print("\n🏆 STAGE 5: EVALUASI AKHIR")
    
    # Menghitung QWK langsung dari hasil ekstraksi, tidak perlu memanggil model lagi
    pred_classes = np.round(np.array(test_preds) * 2).astype(int)
    true_classes = np.round(np.array(test_targets) * 2).astype(int)
    
    final_qwk = cohen_kappa_score(true_classes, pred_classes, weights='quadratic')

    print("="*50)
    print(f"🌟 SKOR QWK FINAL (PADA TEST SET MURNI) UNTUK TRAIT '{TARGET_TRAIT.upper()}': {final_qwk:.4f} 🌟")
    print("="*50)

c:\Users\gabyg\miniconda3\envs\skripsi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔥 Using Device: xpu
🎯 Fokus Pelatihan: Trait CLARITY
🔄 Menemukan train_df_v2.csv dan test_df_v2.csv! Memuat data eksisting agar Test Set murni tetap sama...

✅ Data Total: 1054 | Train Murni: 867 | Test Murni: 187

🚀 STAGE 2: MENCARI PARAMETER TERBAIK (CROSS-VALIDATION)

⚙️ [1/12] Uji: LR=1e-05 | Drop=0.1 | Epochs=10


Loading weights: 100%|██████████| 197/197 [00:01<00:00, 174.67it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


   Fold 1 QWK: 0.2374


Loading weights: 100%|██████████| 197/197 [00:01<00:00, 164.30it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

   Fold 3 QWK: 0.3040
📊 Rata-rata QWK: 0.2164
💾 Progress aman! Tersimpan ke grid_search_clarity_results.csv


🏆 HASIL GRID SEARCH SELESAI!
|   combo_index |   learning_rate |   dropout_rate |   epochs |   Mean_QWK |
|--------------:|----------------:|---------------:|---------:|-----------:|
|             0 |           1e-05 |            0.1 |       10 |   0.218808 |
|             1 |           1e-05 |            0.1 |       15 |   0.259196 |
|             2 |           1e-05 |            0.2 |       10 |   0.194343 |
|             3 |           1e-05 |            0.2 |       15 |   0.292529 |
|             4 |           2e-05 |            0.1 |       10 |   0.146151 |
|             5 |           2e-05 |            0.1 |       15 |   0.215689 |
|             6 |           2e-05 |            0.2 |       10 |   0.286864 |
|             7 |           2e-05 |            0.2 |       15 |   0.216464 |
|             8 |           3e-05 |            0.1 |       10 |   0.241922 |
|             

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 610.48it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Final Epoch 15/15: 100%|██████████| 434/434 [03:22<00:00,  2.14it/s, Loss=0.7385]


✅ Final Model Tersimpan: best_roberta_clarity_final.pth

⚙️ STAGE 4: EXTRACTING ROBERTA PREDICTIONS (MENYIMPAN KE CSV)...


Extracting Test Preds: 100%|██████████| 94/94 [00:13<00:00,  7.09it/s]



✅ Prediksi Train Set disimpan ke: roberta_train_clarity.csv
✅ Prediksi Test Set disimpan ke : roberta_test_clarity.csv

🏆 STAGE 5: EVALUASI AKHIR
🌟 SKOR QWK FINAL (PADA TEST SET MURNI) UNTUK TRAIT 'CLARITY': 0.3620 🌟
